# 11 — Direction 1, step B: a reduced-order model of the thermal core

`hvac_controller/d1/rom.py` takes each core saved by notebook 10 (matrices + the logged 4-day baseline rollout) and:

1. re-runs the exact core from the saved arrays against step A's swap-free replay (`loader_err_k`: must be ~1e-12);
2. builds a block rational Krylov basis in the capacitance inner product, shifts sigma = 1 (one step: A^-1 P, so the
   air-side S is exact), 0 (steady state), 1/12 and 1/288 (hours, a day), and projects (Galerkin, stable at any order);
3. diagonalises it (modal ROM) and truncates it to 400 / 200 / 100 / 50 / 25 states by modal dominance and by balanced
   truncation (feedthrough keeping either the steady state, `balanced_dc`, or the one-step response, `balanced_step`);
4. replays every ROM on the logged rollout (vs the exact core and vs the real sensors, which include the mixing swap)
   and on a random excitation with building-wide air-off blocks (vs the exact core, zone means).

**Setup:** Internet on (clones the repo). **Add Input → Your Work → notebook `10_d1_core_check` (its output)**; no
competition data needed. CPU. Expected ≈ 10–15 min (the three buildings run in parallel).
**Gate (D1 doc §5, step 2):** zone MAE < 0.1 K at 24 h with ≤ 1,000 states. Output: `/kaggle/working/d1_rom.zip`.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
N_JOBS = 3        # buildings built in parallel (4 CPUs; SuperLU is single-threaded)
N_RANDOM = 1152   # steps of random excitation per building (4 days)

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

REPO = Path("/tmp/hvac-controller")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

# Cores saved by notebook 10: attached notebook output keeps the d1_core/ folder; fall back to its zip.
cores = sorted({p.parent for p in Path("/kaggle/input").rglob("vectors.npz")})
if not cores:
    unpacked = Path("/kaggle/temp/d1_core")
    for z in Path("/kaggle/input").rglob("d1_core.zip"):
        zipfile.ZipFile(z).extractall(unpacked)
        break
    cores = sorted({p.parent for p in unpacked.rglob("vectors.npz")})
cores = list({c.name: c for c in cores}.values())  # one per tag
for c in cores:
    print(c)
assert cores, "attach the output of notebook 10_d1_core_check"

In [ ]:
OUT = Path("/kaggle/working/d1_rom")
OUT.mkdir(parents=True, exist_ok=True)
threads = str(max(1, (os.cpu_count() or 4) // N_JOBS))
env = dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS=threads, OPENBLAS_NUM_THREADS=threads,
           MKL_NUM_THREADS=threads)
t0 = time.time()
pending, running = list(cores), []
while pending or running:
    while pending and len(running) < N_JOBS:
        c = pending.pop(0)
        (OUT / c.name).mkdir(exist_ok=True)
        log = open(OUT / c.name / "log.txt", "w")
        cmd = [sys.executable, "-m", "hvac_controller.d1.rom", str(c), str(OUT / c.name), "--n-random", str(N_RANDOM)]
        running.append((c, subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log))
        print(f"[{time.time() - t0:5.0f}s] started {c.name}", flush=True)
    for item in list(running):
        c, proc, log = item
        if proc.poll() is not None:
            log.close()
            running.remove(item)
            print(f"[{time.time() - t0:5.0f}s] {c.name} exit {proc.returncode}", flush=True)
    time.sleep(5)
for c in cores:
    print(f"===== {c.name}")
    print((OUT / c.name / "log.txt").read_text()[-6000:])

In [ ]:
reports = {}
for c in cores:
    f = OUT / c.name / "report.json"
    reports[c.name] = json.loads(f.read_text()) if f.exists() else {"error": "no report (see log above)"}
(OUT / "reports.json").write_text(json.dumps({"commit": COMMIT, "reports": reports}, indent=1))
with zipfile.ZipFile("/kaggle/working/d1_rom.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(OUT.parent))
print("wrote /kaggle/working/d1_rom.zip")

for tag, r in reports.items():
    if "error" in r:
        print(tag, "ERROR", r["error"])
        continue
    print(f"\n{tag}: n={r['n_solved']} mass={r['n_mass']} zones={r['n_zones']} h={r['h']}  loader {r['loader_err_k']:.1e} K"
          f"  r={r['variants'][0]['order']} (dropped {r['basis']['dropped']})  E_r {r['e_r_err']:.0e}  S {r['s_rel_err']:.0e}"
          f"  lambda_raw_min {r['lambda_raw_min']:.2e}  slowest tau {r['slowest_time_constant_h']:.0f} h  {r['seconds']:.0f}s")
    print(f"  {'variant':>18} {'r':>4} | {'log MAE@24h':>11} {'log max@4d':>10} {'sens max@4d':>11} |"
          f" {'rnd MAE@24h':>11} {'rnd max@4d':>10} | {'S err':>7} {'DC err':>7} {'us/step':>7}")
    best = None
    for v in r["variants"]:
        lg = v["logged_vs_core"]; rs = v["logged_vs_sensors"]; rz = v["random_vs_core_zones"]
        last_l, last_r = list(lg)[-1], list(rz)[-1]
        print(f"  {v['name']:>18} {v['order']:4d} | {lg['288_steps']['mae']:11.4f} {lg[last_l]['max']:10.4f}"
              f" {rs[last_l]['max']:11.4f} | {rz['288_steps']['mae']:11.4f} {rz[last_r]['max']:10.4f} |"
              f" {v['s_rel_err']:7.0e} {v['dc_gain_rel_err']:7.0e} {v['step_us']:7.0f}")
        ok = v["order"] <= 1000 and max(lg["288_steps"]["mae"], rz["288_steps"]["mae"]) < 0.1
        if ok and (best is None or v["order"] < best["order"]):
            best = v
    print("  GATE (MAE < 0.1 K at 24 h, <= 1000 states):",
          f"PASS, smallest {best['name']}" if best else "FAIL")